In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q \
transformers==4.52.4 \
accelerate \
sentencepiece \
youtube-transcript-api \
fastapi \
uvicorn \
pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 76.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 77.3 MB/s eta 0:00:00:00:01


In [87]:
import torch
import socket
import threading
import time
import os
from urllib.parse import urlparse, parse_qs

from youtube_transcript_api import YouTubeTranscriptApi

from transformers import pipeline
from transformers import AutoTokenizer

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse

from pyngrok import ngrok, conf

import uvicorn
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [88]:
MODEL_NAME = "facebook/bart-large-cnn"

device = 0 if torch.cuda.is_available() else -1
summarizer = pipeline(
    "summarization",
    model=MODEL_NAME,
    device=device
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Model Loaded Successfully")

Device set to use cuda:0


Model Loaded Successfully


In [106]:
def extract_video_id(url):
    parsed = urlparse(url)
    if "youtu.be" in parsed.netloc:
        return parsed.path.lstrip("/")
    qs = parse_qs(parsed.query)
    ids = qs.get("v")
    if not ids:
        raise ValueError("Invalid YouTube URL")
    return ids[0]

In [107]:
def get_transcript(url):
    video_id = extract_video_id(url)
    api = YouTubeTranscriptApi()
    try:
        transcript = api.fetch(video_id, languages=["en"])
    except Exception:
        raise ValueError(
            "This video doesn't have an English transcript available. "
            "Please try a different video."
        )
    text = "\n".join(snippet.text for snippet in transcript)
    return text

In [108]:
import re

def clean_transcript(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\b(\w+)( \1\b)+', r'\1', text, flags=re.IGNORECASE)
    return text.strip()

In [109]:
def chunk_text(text, max_tokens=900):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(tokenizer.encode(sentence, add_special_tokens=False))

        if current_length + sentence_length > max_tokens and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_length = 0

        current_chunk.append(sentence)
        current_length += sentence_length

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [110]:
def summarize_chunks(chunks):
    summaries = []
    for chunk in chunks:
        input_length = len(tokenizer.encode(chunk, add_special_tokens=False))

        dynamic_max = min(220, max(60, int(input_length * 0.6)))
        dynamic_min = min(100, max(30, int(dynamic_max * 0.5)))

        result = summarizer(
            chunk,
            max_length=dynamic_max,
            min_length=dynamic_min,
            length_penalty=1.0,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )
        summaries.append(result[0]["summary_text"])
    return summaries

In [111]:
def remove_duplicate_sentences(text):

    sentences = text.split(".")

    seen = set()

    cleaned = []

    for sentence in sentences:

        sentence = sentence.strip()

        if sentence and sentence not in seen:

            cleaned.append(sentence)

            seen.add(sentence)

    return ". ".join(cleaned)

In [112]:
def final_summary(summaries):
    combined = " ".join(summaries)
    combined = remove_duplicate_sentences(combined)

    combined_length = len(tokenizer.encode(combined, add_special_tokens=False))

    if combined_length < 700:
        return combined

    combined_chunks = chunk_text(combined, max_tokens=900)

    final_parts = []
    for chunk in combined_chunks:
        input_length = len(tokenizer.encode(chunk, add_special_tokens=False))

        dynamic_max = min(350, max(100, int(input_length * 0.6)))
        dynamic_min = min(180, max(50, int(dynamic_max * 0.5)))

        result = summarizer(
            chunk,
            max_length=dynamic_max,
            min_length=dynamic_min,
            length_penalty=1.0,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )
        final_parts.append(result[0]["summary_text"])

    return " ".join(final_parts)

In [113]:
MAX_CHUNKS = 15 

def summarize_youtube(url):
    transcript = get_transcript(url)
    transcript = clean_transcript(transcript)

    if not transcript.strip():
        raise ValueError("No speech content found in this video.")

    chunks = chunk_text(transcript)

    if len(chunks) > MAX_CHUNKS:
        raise ValueError(
            f"This video is too long to summarize (needs {len(chunks)} chunks, "
            f"max supported is {MAX_CHUNKS}). Try a shorter video."
        )

    summaries = summarize_chunks(chunks)
    return final_summary(summaries)

In [114]:
from fastapi import FastAPI, HTTPException, Header
from pydantic import BaseModel
from fastapi.responses import JSONResponse

In [115]:
NGROK_TOKEN = "3Gm6wzye5DrlvqoOf4r0Eq8zich_3UP6mVEr2Z7SXUh7jn7Ye"

API_KEY = "secret123"

In [116]:
class SummarizeRequest(BaseModel):
    youtube_url: str

app = FastAPI(
    title="YouTube AI Summarizer",
    version="1.0"
)

@app.get("/")
def home():
    return {
        "status": "running",
        "model": MODEL_NAME
    }

In [117]:
@app.post("/summarize")
async def summarize(
    request: SummarizeRequest,
    authorization: str = Header(None)
):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    try:
        summary = summarize_youtube(request.youtube_url)
        return JSONResponse({
            "status": "success",
            "summary": summary
        })
    except ValueError as e:
        return JSONResponse(
            status_code=400,
            content={"status": "error", "message": str(e)}
        )
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"status": "error", "message": str(e)}
        )

In [118]:
from pyngrok import ngrok, conf
import socket
import threading
import uvicorn
import time

In [119]:
def free_port():

    s = socket.socket()

    s.bind(('',0))

    port = s.getsockname()[1]

    s.close()

    return port

In [120]:
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()  

port = free_port()
public_url = ngrok.connect(port).public_url
print("="*60)
print("PUBLIC URL")
print(public_url)
print("="*60)

def run():
    uvicorn.run(app, host="0.0.0.0", port=port)

threading.Thread(target=run, daemon=True).start()
time.sleep(2)

PUBLIC URL
https://coherent-endowment-outpost.ngrok-free.dev


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:43557 (Press CTRL+C to quit)


In [121]:
import requests

headers = {
    "Authorization": f"Bearer {API_KEY}"
}

payload = {
    "youtube_url": "https://www.youtube.com/watch?v=LScqH7t3rSM"
}

response = requests.post(
    f"{public_url}/summarize",
    headers=headers,
    json=payload
)

print(response.status_code)
print(response.json())

INFO:     34.6.166.197:0 - "POST /summarize HTTP/1.1" 200 OK
200
{'status': 'success', 'summary': '"I had to stop thinking normal. I can no longer be a common man walking around doing common things," he says. "We all have a dungeon. I\'m just willing to talk about mine" "This isn\'t for the masses. This is for those who are leaving normal behind" "You have to make the decision whether you want to be a badass or a mediocre and everything\'s okay" "It\'s so easy to be great nowadays, my friend, cuz most people are weak" "I was at the all-time low and I wasn\'t going anywhere and I was exactly what everybody said I was going to be, which was nothing" "I became obsessed with just being obsessed and that\'s what it was, man. I saw some guys going through Navy Seal training and they were going through hell week and they\'re getting their ass just beat, you know, in and out of the water" "You\'re always a victim. Even if you have everything in life, until you realize that you\'ve cheated" Gag